# Working with timezones

A timestamp without a timezone is ambiguous when data comes from multiple regions. This notebook shows how to treat UTC as a common reference, convert to local time, and derive local calendar features safely.

## The key idea

The same instant can have different local dates and hours. Store or compare events in a common timezone such as UTC, then convert to the business or customer timezone before extracting local hour, weekday, or weekend. A naive timestamp should not be silently interpreted as UTC or local time without a documented rule.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid',context='notebook')

events=pd.DataFrame({'event_id':[1,2,3],
    'event_utc':pd.to_datetime(['2024-01-15 14:00:00+00:00','2024-07-15 14:00:00+00:00','2024-03-10 07:30:00+00:00'],utc=True),
    'customer_timezone':['America/New_York','Asia/Kolkata','America/New_York']})
events

## 1. Convert one instant to local time

Use a named IANA timezone such as America/New_York rather than a fixed abbreviation. Named zones include daylight-saving rules.

In [ ]:
events['event_local']=events.apply(lambda row:row['event_utc'].tz_convert(row['customer_timezone']),axis=1)
events['local_date']=events['event_local'].map(lambda value:value.date())
events['local_hour']=events['event_local'].map(lambda value:value.hour)
events['local_day_name']=events['event_local'].map(lambda value:value.day_name())
events['local_is_weekend']=events['event_local'].map(lambda value:value.weekday() >= 5)
display(events)

## 2. A daylight-saving transition

On a daylight-saving transition, some local clock times do not exist or occur twice. Converting timezone-aware UTC instants avoids inventing an impossible local time.

In [ ]:
dst=pd.DataFrame({'utc_time':pd.to_datetime(['2024-03-10 06:30:00+00:00','2024-03-10 07:30:00+00:00'],utc=True)})
dst['new_york_time']=dst['utc_time'].dt.tz_convert('America/New_York')
dst['new_york_hour']=dst['new_york_time'].dt.hour
display(dst)
print('Notice that the local clock moves from 01:30 to 03:30 in this example.')

## 3. Convert a column with one known timezone

When every row uses the same source timezone, use tz_convert on the entire Series. If each row has a different customer timezone, convert per row as demonstrated above, or group by timezone for larger datasets.

In [ ]:
utc_times=pd.Series(pd.date_range('2024-05-01 12:00',periods=6,freq='6h',tz='UTC'),name='utc_time')
local_times=utc_times.dt.tz_convert('Europe/Oslo')
time_view=pd.DataFrame({'utc_time':utc_times,'oslo_time':local_times})
time_view['oslo_hour']=time_view['oslo_time'].dt.hour
display(time_view)

## Practical rules

1. Decide what timezone the source system uses.
2. Parse with an explicit timezone policy.
3. Store event instants in UTC when possible.
4. Convert to local time before deriving local calendar features.
5. Keep the original UTC instant for ordering and auditing.
6. Test daylight-saving boundaries and dates near midnight.
7. Do not compare naive timestamps from different regions.

## Summary

Timezone handling is part of feature correctness, not just display formatting. A wrong timezone can change the day, weekend flag, hour, and elapsed duration, which can silently damage a model.